# Project: Atlas, a Resumable Coworker

**What you will build:** a small coworker that reads a project workspace, runs Python to calculate task metrics, writes a weekly report, and prepares an external update. Risky actions pause for approval. The run can be reconstructed from SQLite, and retrying after a crash does not create a second send.

The coding agent in 0.7 works vertically on source code. Atlas works horizontally across notes, meetings, a CSV, approvals, and a business action. The agent loop is still small; this project builds the durable runtime around it.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/32c_project_resumable_coworker.ipynb), or run locally in Jupyter. The complete offline path uses only the Python standard library.


## The contract

Atlas receives one request:

> Review the project brief, meeting notes, and `tasks.csv`. Calculate the workload and blockers, save a weekly report, then prepare an update for the team.

Six tools are enough:

| Tool | Risk | Policy |
|---|---|---|
| `list_files`, `read_file`, `search_workspace` | read | run automatically |
| `run_python` | execute | wait for a person |
| `save_report` | bounded local write | run automatically |
| `send_update` | external effect | wait for a person |

The external send is a local **Outbox**, not Slack or email. That keeps the project testable and puts the difficult part in view: what happens if the effect succeeds and the process dies before the tool result is saved?


```text
model ──tool call──▶ policy ──▶ pending approval ──▶ executor
  ▲                         SQLite                    │
  └──────────── tool result / resume ◀───────────────┘
                                        │
                                  effect ledger
                                        │
                                     Outbox
```

We will keep three kinds of state separate:

- **Chat state:** model-facing messages stored as events.
- **Execution state:** pending approvals and the position implied by unanswered tool calls.
- **Business state:** reports and queued updates. The effect ledger protects their boundary.


## 1. Create the project workspace

The fixture is deliberately small enough to inspect by eye. Re-running this cell resets the demo.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import sqlite3
import subprocess
import sys
import tempfile
import time
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any, Callable

DEMO_ROOT = Path("atlas_coworker_demo").resolve()
WORKSPACE = DEMO_ROOT / "workspace"
DB_PATH = DEMO_ROOT / "runtime.sqlite3"
if DEMO_ROOT.exists():
    shutil.rmtree(DEMO_ROOT)
(WORKSPACE / "meetings").mkdir(parents=True)
(WORKSPACE / "reports").mkdir()

fixtures = {
    "brief.md": """# Atlas

Ship the reporting beta on 31 July. Legal review and the data export are the current risks.
""",
    "meetings/2026-07-23.md": """# Weekly sync

Maya will finish the dashboard. Leo is waiting for Legal. Noor needs the export schema from Maya.
""",
    "tasks.csv": """id,owner,status,task
T-101,Maya,in_progress,Finish dashboard
T-102,Leo,blocked,Obtain legal approval
T-103,Maya,done,Confirm beta scope
T-104,Noor,blocked,Validate export schema
T-105,Leo,in_progress,Prepare release notes
""",
}
for name, content in fixtures.items():
    (WORKSPACE / name).write_text(content, encoding="utf-8")
print(WORKSPACE)


## 2. Build narrow tools

File access goes through one path resolver. `save_report` is more restricted than a general write tool: it only accepts Markdown below `reports/`. That restriction is why our policy can allow it without stopping the run.


In [ ]:
def workspace_path(relative: str) -> Path:
    path = (WORKSPACE / relative).resolve()
    try:
        path.relative_to(WORKSPACE)
    except ValueError as exc:
        raise ValueError("Path escapes the Atlas workspace") from exc
    return path


def list_files(pattern: str = "**/*") -> str:
    files = sorted(
        str(path.relative_to(WORKSPACE))
        for path in WORKSPACE.glob(pattern) if path.is_file()
    )
    return "\n".join(files) or "(no files)"


def read_file(path: str) -> str:
    return workspace_path(path).read_text(encoding="utf-8")


def search_workspace(query: str) -> str:
    hits = []
    for path in sorted(WORKSPACE.rglob("*")):
        if not path.is_file():
            continue
        for number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
            if query.lower() in line.lower():
                hits.append(f"{path.relative_to(WORKSPACE)}:{number}: {line}")
    return "\n".join(hits) or "(no matches)"


def save_report(path: str, content: str) -> str:
    target = workspace_path(path)
    try:
        target.relative_to((WORKSPACE / "reports").resolve())
    except ValueError as exc:
        raise ValueError("Reports must stay below reports/") from exc
    if target.suffix != ".md":
        raise ValueError("Reports must use the .md extension")
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
    return f"Saved {target.relative_to(WORKSPACE)}"


### Python execution

Unlike the in-process `exec` used by the Data Analyst project, this tool starts a fresh Python process. It copies CSV inputs into a temporary directory, uses isolated Python mode, limits runtime, and truncates output. Each call is reproducible and leaves no notebook variables behind.


In [ ]:
def run_python(code: str) -> str:
    """Run one isolated Python process over copied CSV inputs."""
    with tempfile.TemporaryDirectory(prefix="atlas-python-") as folder:
        folder = Path(folder)
        for csv_file in WORKSPACE.glob("*.csv"):
            shutil.copy2(csv_file, folder / csv_file.name)
        try:
            result = subprocess.run(
                [sys.executable, "-I", "-c", code],
                cwd=folder,
                env={"PATH": os.environ.get("PATH", ""), "PYTHONIOENCODING": "utf-8"},
                capture_output=True,
                text=True,
                timeout=5,
            )
        except subprocess.TimeoutExpired:
            return "Error: Python timed out after 5 seconds"

    parts = [f"exit_code={result.returncode}"]
    if result.stdout.strip():
        parts.append("stdout:\n" + result.stdout.strip()[:4_000])
    if result.stderr.strip():
        parts.append("stderr:\n" + result.stderr.strip()[:4_000])
    return "\n".join(parts)


print(run_python(
    "import csv; rows=list(csv.DictReader(open('tasks.csv'))); "
    "print('blocked=', sum(r['status']=='blocked' for r in rows), sep='')"
))


```{warning}
This is **process isolation, not a security sandbox**. The child process still inherits the operating-system permissions of the notebook user and may have network access. Approval answers whether the code should run; a container, VM, OS sandbox, or hosted code interpreter limits what approved code can do. Do not expose this executor to untrusted users as written.
```

There is no general shell tool in the core project. The subprocess already teaches execution controls, while 0.7 covers shell parsing, command policy, and why `cwd` is not containment.


## 3. Describe capabilities and risk

The runtime works with canonical tool calls rather than provider-specific objects. `ScriptedProvider` uses the persisted assistant-message count as its cursor, so creating a new provider object after a restart does not lose the trajectory.


In [ ]:
class Risk(str, Enum):
    READ = "read"
    WRITE_LOCAL = "write_local"
    EXEC = "exec"
    EXTERNAL = "external"


@dataclass(frozen=True)
class ToolCall:
    id: str
    name: str
    arguments: dict[str, Any]


@dataclass(frozen=True)
class AssistantTurn:
    content: str = ""
    tool_calls: list[ToolCall] = field(default_factory=list)


@dataclass(frozen=True)
class Tool:
    handler: Callable[..., str]
    risk: Risk


class PermissionPolicy:
    def needs_review(self, tool: Tool) -> bool:
        return tool.risk in {Risk.EXEC, Risk.EXTERNAL}


class ScriptedProvider:
    """A deterministic model double whose cursor lives in the transcript."""

    def __init__(self, script: list[AssistantTurn]):
        self.script = script

    def complete(self, messages: list[dict]) -> AssistantTurn:
        index = sum(message.get("role") == "assistant" for message in messages)
        if index >= len(self.script):
            raise RuntimeError("Script exhausted")
        return self.script[index]


In [ ]:
class ToolRegistry:
    def __init__(self):
        self.tools: dict[str, Tool] = {}

    def register(self, name: str, handler: Callable[..., str], risk: Risk) -> None:
        self.tools[name] = Tool(handler, risk)

    def get(self, name: str) -> Tool:
        if name not in self.tools:
            raise KeyError(f"Unknown tool: {name}")
        return self.tools[name]

    def execute(self, call: ToolCall) -> str:
        return str(self.get(call.name).handler(**call.arguments))


registry = ToolRegistry()
registry.register("list_files", list_files, Risk.READ)
registry.register("read_file", read_file, Risk.READ)
registry.register("search_workspace", search_workspace, Risk.READ)
registry.register("run_python", run_python, Risk.EXEC)
registry.register("save_report", save_report, Risk.WRITE_LOCAL)
# The runtime handles this external effect through the ledger.
registry.register("send_update", lambda **_: "unreachable", Risk.EXTERNAL)

print([(name, tool.risk.value) for name, tool in registry.tools.items()])


## 4. Persist execution as events

SQLite has four responsibilities here:

| Storage | Meaning |
|---|---|
| `events` | append-only messages, tool lifecycle, and metrics |
| `pending_actions` | approval inbox and resume state |
| `effect_ledger` | completed external calls keyed by session and tool-call ID |
| `outbox` | simulated business delivery |

There is no separate program-counter column. If an assistant tool call has no tool result, the runtime knows exactly what remains to be handled.


In [ ]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


class EventStore:
    def __init__(self, path: Path):
        self.db = sqlite3.connect(path)
        self.db.row_factory = sqlite3.Row
        self.db.executescript("""
            CREATE TABLE IF NOT EXISTS events (
                seq INTEGER PRIMARY KEY, session_id TEXT, kind TEXT,
                payload TEXT, created_at TEXT
            );
            CREATE TABLE IF NOT EXISTS pending_actions (
                session_id TEXT, call_id TEXT, tool_name TEXT, arguments TEXT,
                status TEXT, reason TEXT, PRIMARY KEY (session_id, call_id)
            );
            CREATE TABLE IF NOT EXISTS effect_ledger (
                session_id TEXT, call_id TEXT, result TEXT,
                PRIMARY KEY (session_id, call_id)
            );
            CREATE TABLE IF NOT EXISTS outbox (
                id INTEGER PRIMARY KEY, session_id TEXT, call_id TEXT,
                recipient TEXT, message TEXT, UNIQUE (session_id, call_id)
            );
        """)

    def emit(self, session: str, kind: str, payload: dict) -> None:
        with self.db:
            self.db.execute(
                "INSERT INTO events(session_id, kind, payload, created_at) VALUES (?, ?, ?, ?)",
                (session, kind, json.dumps(payload), utc_now()),
            )

    def message(self, session: str, value: dict) -> None:
        self.emit(session, "message", {"message": value})

    def messages(self, session: str) -> list[dict]:
        rows = self.db.execute(
            "SELECT payload FROM events WHERE session_id=? AND kind='message' ORDER BY seq",
            (session,),
        )
        return [json.loads(row["payload"])["message"] for row in rows]

    def events(self, session: str) -> list[dict]:
        rows = self.db.execute(
            "SELECT kind, payload FROM events WHERE session_id=? ORDER BY seq", (session,)
        )
        return [{"kind": row["kind"], "payload": json.loads(row["payload"])} for row in rows]

    def pending(self, session: str, call_id: str) -> dict | None:
        row = self.db.execute(
            "SELECT * FROM pending_actions WHERE session_id=? AND call_id=?",
            (session, call_id),
        ).fetchone()
        return dict(row) if row else None

    def request_approval(self, session: str, call: ToolCall) -> dict:
        with self.db:
            cursor = self.db.execute(
                "INSERT OR IGNORE INTO pending_actions VALUES (?, ?, ?, ?, 'pending', '')",
                (session, call.id, call.name, json.dumps(call.arguments)),
            )
        if cursor.rowcount:
            self.emit(session, "approval_required", {"call_id": call.id, "tool": call.name})
        return self.pending(session, call.id)

    def decide(self, session: str, call_id: str, approved: bool, reason: str) -> None:
        status = "approved" if approved else "denied"
        with self.db:
            cursor = self.db.execute(
                "UPDATE pending_actions SET status=?, reason=? "
                "WHERE session_id=? AND call_id=? AND status='pending'",
                (status, reason, session, call_id),
            )
        if cursor.rowcount != 1:
            raise ValueError("Approval is missing or already decided")
        self.emit(session, "approval_decided", {"call_id": call_id, "approved": approved})

    def finish_action(self, session: str, call_id: str) -> None:
        with self.db:
            self.db.execute(
                "UPDATE pending_actions SET status='completed' WHERE session_id=? AND call_id=?",
                (session, call_id),
            )

    def send_once(self, session: str, call: ToolCall) -> tuple[str, bool]:
        self.db.execute("BEGIN IMMEDIATE")
        try:
            row = self.db.execute(
                "SELECT result FROM effect_ledger WHERE session_id=? AND call_id=?",
                (session, call.id),
            ).fetchone()
            if row:
                self.db.commit()
                return row["result"], True
            result = f"Queued update for {call.arguments['recipient']}"
            self.db.execute(
                "INSERT INTO outbox(session_id, call_id, recipient, message) VALUES (?, ?, ?, ?)",
                (session, call.id, call.arguments["recipient"], call.arguments["message"]),
            )
            self.db.execute(
                "INSERT INTO effect_ledger VALUES (?, ?, ?)", (session, call.id, result)
            )
            self.db.commit()
            return result, False
        except Exception:
            self.db.rollback()
            raise

    def rows(self, table: str) -> list[dict]:
        if table not in {"pending_actions", "effect_ledger", "outbox"}:
            raise ValueError("Unknown table")
        return [dict(row) for row in self.db.execute(f"SELECT * FROM {table}")]


## 5. Drive, pause, and resume

`CoworkerRuntime` has no `input()`. When policy requires a person, it writes a pending action and returns control to the caller. A notebook, API, queue worker, or UI can make the decision later.

Tool results carry a small `_display` sidecar for the application. `provider_messages` removes it before the transcript reaches a model.


In [ ]:
DISPLAY_ONLY = {"_display"}


def provider_feed(messages: list[dict]) -> list[dict]:
    return [
        {key: value for key, value in message.items() if key not in DISPLAY_ONLY}
        for message in messages
    ]


def assistant_message(turn: AssistantTurn) -> dict:
    message = {"role": "assistant", "content": turn.content or None}
    if turn.tool_calls:
        message["tool_calls"] = [{
            "id": call.id,
            "type": "function",
            "function": {"name": call.name, "arguments": json.dumps(call.arguments)},
        } for call in turn.tool_calls]
    return message


def unanswered_calls(messages: list[dict]) -> list[ToolCall]:
    answered = {m["tool_call_id"] for m in messages if m.get("role") == "tool"}
    return [
        ToolCall(raw["id"], raw["function"]["name"], json.loads(raw["function"]["arguments"]))
        for message in messages for raw in message.get("tool_calls", [])
        if raw["id"] not in answered
    ]


@dataclass(frozen=True)
class RunResult:
    status: str
    message: str = ""
    pending: dict | None = None


class SimulatedCrash(RuntimeError):
    pass


class CoworkerRuntime:
    def __init__(
        self, provider, registry, policy, store,
        max_model_turns: int = 12, crash_after_effect: str | None = None,
    ):
        self.provider, self.registry = provider, registry
        self.policy, self.store = policy, store
        self.max_model_turns = max_model_turns
        self.crash_after_effect = crash_after_effect

    def start(self, session: str, task: str) -> RunResult:
        if self.store.messages(session):
            raise ValueError(f"Session already exists: {session}")
        self.store.message(session, {"role": "user", "content": task})
        return self.run(session)

    def resume(
        self, session: str, call_id: str, approved: bool, reason: str = ""
    ) -> RunResult:
        self.store.decide(session, call_id, approved, reason)
        return self.run(session)

    def run(self, session: str) -> RunResult:
        while True:
            messages = self.store.messages(session)
            pending_calls = unanswered_calls(messages)
            if pending_calls:
                for call in pending_calls:
                    paused = self._execute_or_pause(session, call)
                    if paused:
                        return paused
                continue

            turns = sum(m.get("role") == "assistant" for m in messages)
            if turns >= self.max_model_turns:
                return RunResult("max_model_turns", "Model turn budget exhausted")
            turn = self.provider.complete(provider_feed(messages))
            self.store.message(session, assistant_message(turn))
            self.store.emit(session, "model_turn", {"has_tools": bool(turn.tool_calls)})
            if not turn.tool_calls:
                return RunResult("complete", turn.content)

    def _execute_or_pause(self, session: str, call: ToolCall) -> RunResult | None:
        try:
            tool = self.registry.get(call.name)
        except KeyError as exc:
            self._result(session, call, f"Error: {exc}", "unknown")
            return None

        action = self.store.pending(session, call.id)
        if self.policy.needs_review(tool):
            action = action or self.store.request_approval(session, call)
            if action["status"] == "pending":
                return RunResult("waiting_for_approval", call.name, action)
            if action["status"] == "denied":
                self._result(session, call, f"Denied: {action['reason']}", tool.risk.value)
                self.store.finish_action(session, call.id)
                return None

        started = time.perf_counter()
        try:
            reused = False
            if tool.risk is Risk.EXTERNAL:
                output, reused = self.store.send_once(session, call)
                if self.crash_after_effect == call.id and not reused:
                    raise SimulatedCrash("Effect committed before tool result")
            else:
                output = self.registry.execute(call)
        except SimulatedCrash:
            raise
        except Exception as exc:
            output = f"Error: {type(exc).__name__}: {exc}"

        if reused:
            self.store.emit(session, "effect_reused", {"call_id": call.id})
        self._result(
            session, call, output, tool.risk.value,
            round((time.perf_counter() - started) * 1000, 2),
        )
        if action:
            self.store.finish_action(session, call.id)
        return None

    def _result(
        self, session: str, call: ToolCall, output: str, risk: str, duration_ms: float = 0,
    ) -> None:
        self.store.message(session, {
            "role": "tool", "tool_call_id": call.id, "content": output,
            "_display": {"tool": call.name, "risk": risk},
        })
        self.store.emit(session, "tool_finished", {
            "call_id": call.id, "tool": call.name,
            "duration_ms": duration_ms, "error": output.startswith("Error:"),
        })


## 6. Give Atlas a deterministic trajectory

The provider below stands in for a model. It reads evidence, asks Python for exact counts, writes the report, and proposes the update. Because the decisions are scripted, failures in this notebook point to the runtime rather than to model randomness.

This is the same role played by `FunctionModel` and `TestModel` in 1.5b, expressed at the provider boundary introduced in 0.9.


In [ ]:
analysis_code = """
import csv
from collections import Counter
rows = list(csv.DictReader(open("tasks.csv", encoding="utf-8")))
owners = Counter(row["owner"] for row in rows)
print("blocked=" + str(sum(row["status"] == "blocked" for row in rows)))
print("owners=" + ", ".join(f"{name}:{owners[name]}" for name in sorted(owners)))
"""

report = """# Atlas weekly report

## Status
- 5 tracked tasks: 1 done, 2 in progress, 2 blocked.
- Workload: Leo 2, Maya 2, Noor 1.

## Blockers
- Legal approval is blocking Leo.
- The export schema is blocking Noor and depends on Maya.
"""

SCRIPT = [
    AssistantTurn(tool_calls=[ToolCall("read_brief", "read_file", {"path": "brief.md"})]),
    AssistantTurn(tool_calls=[ToolCall(
        "read_meeting", "read_file", {"path": "meetings/2026-07-23.md"}
    )]),
    AssistantTurn(tool_calls=[ToolCall("exec_metrics", "run_python", {"code": analysis_code})]),
    AssistantTurn(tool_calls=[ToolCall(
        "save_weekly", "save_report", {"path": "reports/weekly.md", "content": report}
    )]),
    AssistantTurn(tool_calls=[ToolCall("send_weekly", "send_update", {
        "recipient": "atlas-team@example.com",
        "message": "Atlas has 2 blocked tasks. The weekly report is ready for review.",
    })]),
    AssistantTurn("The report is saved and the approved team update is queued."),
]
TASK = "Review Atlas, calculate blockers, save reports/weekly.md, and prepare a team update."


### Run until Python needs approval

Read-only calls proceed without interruption. Generated code does not.


In [ ]:
store = EventStore(DB_PATH)
runtime = CoworkerRuntime(
    ScriptedProvider(SCRIPT), registry, PermissionPolicy(), store
)

first_stop = runtime.start("atlas-week-30", TASK)
print(first_stop.status)
print(first_stop.pending["tool_name"], first_stop.pending["call_id"])
print(first_stop.pending["arguments"][:240] + "...")

assert first_stop.status == "waiting_for_approval"
assert first_stop.pending["call_id"] == "exec_metrics"


The approval is now data in SQLite. We can discard the runtime and provider objects before deciding. This is the Colab-friendly equivalent of a worker process stopping and starting again.


In [ ]:
store.db.close()
del runtime, store

store = EventStore(DB_PATH)
runtime = CoworkerRuntime(
    ScriptedProvider(SCRIPT), registry, PermissionPolicy(), store
)
restored = store.pending("atlas-week-30", "exec_metrics")
print("restored:", restored["status"], restored["tool_name"])

second_stop = runtime.resume(
    "atlas-week-30", "exec_metrics", approved=True,
    reason="The code only reads the copied tasks.csv fixture.",
)
print("next:", second_stop.status, second_stop.pending["tool_name"])
print("\nREPORT:\n", read_file("reports/weekly.md"))

assert second_stop.status == "waiting_for_approval"
assert second_stop.pending["call_id"] == "send_weekly"


## 7. Crash in the awkward gap

Suppose the reviewer approves the send. Atlas writes the Outbox and effect ledger in one transaction, then the process dies **before** adding the tool result to the transcript.

On recovery, `send_weekly` is still an unanswered call. Running it blindly would duplicate the update. The ledger turns its stable session and tool-call IDs into the idempotency key.


In [ ]:
crashing_runtime = CoworkerRuntime(
    ScriptedProvider(SCRIPT), registry, PermissionPolicy(), store,
    crash_after_effect="send_weekly",
)

try:
    crashing_runtime.resume(
"atlas-week-30", "send_weekly", approved=True,
reason="The report and recipient were reviewed.",
    )
except SimulatedCrash as exc:
    print("SIMULATED CRASH:", exc)

print("outbox after crash:", len(store.rows("outbox")))
print("unanswered after crash:", [
    call.id for call in unanswered_calls(store.messages("atlas-week-30"))
])

assert len(store.rows("outbox")) == 1
assert [call.id for call in unanswered_calls(
    store.messages("atlas-week-30")
)] == ["send_weekly"]


In [ ]:
# A fresh process would reconstruct these three objects from configuration.
store.db.close()
store = EventStore(DB_PATH)
recovered_runtime = CoworkerRuntime(
    ScriptedProvider(SCRIPT), registry, PermissionPolicy(), store
)
completed = recovered_runtime.run("atlas-week-30")

print(completed.status, "-", completed.message)
print("outbox after recovery:", len(store.rows("outbox")))
print("effect ledger:", store.rows("effect_ledger"))

assert completed.status == "complete"
assert len(store.rows("outbox")) == 1
assert unanswered_calls(store.messages("atlas-week-30")) == []


The second execution did not send again. It found `send_weekly` in `effect_ledger`, reused the stored result, completed the pending action, and let the model produce its final response.

Our Outbox and ledger share one SQLite transaction, so the local example can be exact. With a real email or Slack API, the database cannot atomically commit the remote service. Pass the same idempotency key downstream when the service supports it; otherwise delivery is normally **at least once**, and the receiver must tolerate duplicates.


## 8. Test trajectories, not just the final sentence

These assertions check the runtime's promises: no orphaned calls after completion, one business effect, display metadata filtered from the provider feed, path containment, and a denial that still becomes a tool result.


In [ ]:
messages = store.messages("atlas-week-30")
events = Counter(event["kind"] for event in store.events("atlas-week-30"))
assert unanswered_calls(messages) == []
assert len(store.rows("outbox")) == 1
assert len(store.rows("effect_ledger")) == 1
assert events["effect_reused"] == 1
assert all("_display" not in message for message in provider_feed(messages))

try:
    read_file("../outside.txt")
    raise AssertionError("Path escape should fail")
except ValueError:
    pass

denial_script = [
    AssistantTurn(tool_calls=[ToolCall("exec_denied", "run_python", {"code": "print(1)"})]),
    AssistantTurn("The code was denied."),
]
denied = CoworkerRuntime(ScriptedProvider(denial_script), registry, PermissionPolicy(), store)
assert denied.start("denial-test", "Run Python").status == "waiting_for_approval"
denied = CoworkerRuntime(ScriptedProvider(denial_script), registry, PermissionPolicy(), store)
assert denied.resume("denial-test", "exec_denied", False, "Not reviewed").status == "complete"
denied_messages = store.messages("denial-test")
assert unanswered_calls(denied_messages) == []
assert any(m.get("role") == "tool" and m["content"].startswith("Denied:")
           for m in denied_messages)
print("Acceptance checks passed.")


In [ ]:
finished_events = [
    event for event in store.events("atlas-week-30")
    if event["kind"] == "tool_finished"
]
metrics = {
    "model_turns": events["model_turn"],
    "tool_calls": events["tool_finished"],
    "approvals_requested": events["approval_required"],
    "approvals_decided": events["approval_decided"],
    "duplicates_prevented": events["effect_reused"],
    "tool_time_ms": round(sum(
event["payload"]["duration_ms"] for event in finished_events
    ), 2),
}
print(json.dumps(metrics, indent=2))


These are useful portfolio numbers because they describe a run rather than merely claiming that the agent is reliable. A live provider adapter can add tokens, cost, and model latency to the same event stream.


## 9. Rosetta: raw runtime and LangGraph

You already paused and resumed a graph in 2.3–2.4. This project does not replace that machinery; it exposes the application responsibilities around it.

| Atlas implementation | LangGraph concept | What remains yours |
|---|---|---|
| message events | checkpointed graph state | retention and model-facing filtering |
| `pending_actions` | `interrupt()` payload | approval inbox, identity, expiry, authorization |
| `resume(call_id, decision)` | `Command(resume=...)` | API/UI carrying the decision |
| separate tool steps | node boundaries | keeping side effects away from re-executed work |
| `effect_ledger` + Outbox | no automatic external equivalent | idempotency and delivery contract |

A checkpointer restores graph state. **A checkpointer alone cannot prove that an external recipient saw an effect exactly once.** That boundary needs an idempotency key, a transactional Outbox, or cooperation from the downstream service.


:::{dropdown} Optional: connect a live OpenRouter model
:color: info

The durable path is deterministic on purpose. To connect a model without enlarging this project, reuse `OpenRouterProvider` and the JSON tool schemas from 0.9:

1. Give each registered tool its description and JSON input schema.
2. Replace `ScriptedProvider.complete(messages)` with the adapter's model call.
3. Set `parallel_tool_calls=False` so this compact approval runtime handles one proposal at a time.
4. Run the same acceptance tests before treating the live trajectory as trustworthy.

The provider changes; SQLite, policy, approvals, effect ledger, and Outbox do not.
:::


:::{dropdown} Try it yourself
:color: secondary

1. **Edit before approval.** Add `edited_arguments` to `pending_actions` and let a reviewer change the recipient or message. **Done when:** the Outbox contains the reviewed values, not the original proposal.
2. **Expire approvals.** Reject a pending action older than five minutes. **Done when:** resuming an expired call creates a denial tool result and no effect.
3. **Add a model-turn budget test.** Use a script that only requests reads and set `max_model_turns=2`. **Done when:** the third model call never happens and the session reports `max_model_turns`.
4. **Replace the Outbox dispatcher.** Send to a local HTTP test server using `call.id` as an idempotency header. **Done when:** retrying after a simulated crash still produces one accepted request.
5. **Ship the surface.** Expose start, pending approvals, resume, events, and cancel endpoints with the FastAPI pattern from 3.0. Keep the runtime independent of HTTP.
:::


:::{dropdown} Why the shell is optional
:color: info

A general shell would be useful for a desktop coworker, but it does not add the main lesson here. It also brings command parsing, executable allowlists, environment inheritance, and another misleading `cwd` boundary. Reuse the `run_shell` design from 0.7 as an extension and classify it as `Risk.EXEC`; do not treat approval as containment.

Note that arbitrary Python can itself start subprocesses. Omitting a shell narrows the interface and the size of this project, not the operating-system permissions of the child process.
:::


:::{dropdown} Common issues
:color: info

| Symptom | Likely cause | Fix |
|---|---|---|
| `Session already exists` | The start cell was run twice without resetting | Re-run the workspace setup cell |
| `Script exhausted` | The scripted trajectory lacks a final assistant message | End the script with a turn that has no tool calls |
| Approval stays pending | A new runtime was created but `resume(...)` was not called | Pass the same session and call IDs stored in SQLite |
| Two Outbox rows appear | The call ID changed during retry | Preserve provider tool-call IDs across persistence and replay |
| Python cannot import a package | `-I` starts isolated Python and the demo assumes the standard library | Use `csv`, or move to a managed sandbox with explicit dependencies |
| A real API still duplicates delivery | The remote effect cannot share SQLite's transaction | Send an idempotency key and design for at-least-once delivery |
:::


## What you built

Atlas can act, wait without holding a process open, survive reconstruction, resume from a human decision, and prove that its simulated external effect was not duplicated. The provider, policy, store, executor, and surface remain separate, so each can change without hiding the invariants.

For larger implementations of the same ideas, compare [OpenWorker](https://github.com/andrewyng/openworker), the [OpenHands Software Agent SDK](https://github.com/OpenHands/software-agent-sdk), and the serializable approval state in the [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/human_in_the_loop/). They add considerably more product machinery; the loop and ownership boundaries remain recognizable.

**What's next:** 3.3 closes the course with a practical map of when to use raw Python, PydanticAI, LangGraph, MCP, or a larger harness.
